# Regresi Logistik Lanjutan dan Ekstensinya

Catatan ini merupakan pendalaman dari algoritma Regresi Logistik. Meskipun merupakan salah satu algoritma klasifikasi linier yang paling sederhana, Regresi Logistik memiliki berbagai ekstensi dan parameter tingkat lanjut yang membuatnya sangat tangguh untuk berbagai skenario dunia nyata.

Fokus utama pada modul ini adalah penanganan klasifikasi multikelas, penyesuaian bobot untuk data yang tidak seimbang (*imbalanced data*), pemilihan algoritma optimasi (*solver*), serta ekstensi non-linier menggunakan teknik rekayasa fitur polinomial.

#### **Tujuan Pembelajaran**
* Mengimplementasikan dan membedakan pendekatan multikelas: *One-vs-Rest* (OvR) dan *Multinomial* (Softmax).
* Menangani dataset klasifikasi yang tidak seimbang menggunakan parameter `class_weight`.
* Memahami dan memilih *solver* optimasi yang tepat (`lbfgs`, `liblinear`, `saga`) berdasarkan ukuran data dan jenis regularisasi (L1, L2, Elastic-Net).
* Menggabungkan Regresi Logistik dengan `PolynomialFeatures` untuk memecahkan masalah klasifikasi dengan batas keputusan non-linier.

In [24]:
# Memuat pustaka komputasi numerik dan visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Memuat pembuat dataset sintetis dan pemisah data
from sklearn.datasets import make_classification, make_blobs, make_circles
from sklearn.model_selection import train_test_split

# Memuat modul Regresi Logistik, Pipeline, dan Prapemrosesan
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

# Memuat metrik evaluasi
from sklearn.metrics import classification_report, accuracy_score

# Konfigurasi visualisasi
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print("Pustaka untuk Regresi Logistik Lanjutan berhasil dimuat.")

Pustaka untuk Regresi Logistik Lanjutan berhasil dimuat.


## 1. Klasifikasi Multikelas pada Regresi Logistik

Secara bawaan, Regresi Logistik adalah algoritma klasifikasi biner. Namun, pustaka `scikit-learn` memfasilitasi klasifikasi untuk lebih dari dua kelas melalui dua pendekatan matematis:
1. **One-vs-Rest (OvR):** Model memecah masalah multikelas menjadi beberapa klasifikasi biner terpisah (Satu kelas melawan seluruh kelas lainnya).
2. **Multinomial (Softmax):** Model meminimalkan fungsi kerugian multikelas secara serentak yang memprediksi probabilitas probabilitas gabungan seluruh kelas. Pendekatan ini umumnya memberikan performa yang lebih terkalibrasi dan komprehensif.

In [25]:
# Membuat dataset sintetis dengan 4 kelas berbeda
X_multi, y_multi = make_blobs(n_samples=500, centers=4, cluster_std=2.0, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, random_state=42)

# Melatih model dengan pendekatan One-vs-Rest (OvR)
logreg_ovr = LogisticRegression(multi_class='ovr', solver='lbfgs', max_iter=1000)
logreg_ovr.fit(X_train_m, y_train_m)

# Melatih model dengan pendekatan Multinomial
logreg_multi = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
logreg_multi.fit(X_train_m, y_train_m)

print("=== Komparasi Klasifikasi Multikelas ===")
print(f"Akurasi Pendekatan OvR         : {logreg_ovr.score(X_test_m, y_test_m):.4f}")
print(f"Akurasi Pendekatan Multinomial : {logreg_multi.score(X_test_m, y_test_m):.4f}")

=== Komparasi Klasifikasi Multikelas ===
Akurasi Pendekatan OvR         : 0.9680
Akurasi Pendekatan Multinomial : 0.9600


## 2. Penanganan Dataset Tidak Seimbang (*Imbalanced Data*)

Dalam banyak kasus nyata (seperti deteksi penipuan atau diagnosis penyakit), jumlah kelas positif jauh lebih sedikit dibandingkan kelas negatif. Model linier standar cenderung mengabaikan kelas minoritas untuk mencapai akurasi keseluruhan yang tinggi.

Regresi Logistik memiliki parameter adaptif `class_weight='balanced'`. Parameter ini secara otomatis akan memberikan bobot (penalti) yang lebih berat pada kesalahan klasifikasi kelas minoritas, sebanding dengan rasio ketidakseimbangan kelas tersebut.

In [26]:
# Membuat dataset dengan ketidakseimbangan ekstrem (95% kelas 0, 5% kelas 1)
X_imb, y_imb = make_classification(n_samples=1000, n_features=10, weights=[0.95, 0.05], random_state=42)
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(X_imb, y_imb, stratify=y_imb, random_state=42)

# Regresi Logistik Standar (Tanpa pembobotan)
logreg_standard = LogisticRegression(max_iter=1000).fit(X_train_imb, y_train_imb)
pred_standard = logreg_standard.predict(X_test_imb)

# Regresi Logistik dengan class_weight='balanced'
logreg_balanced = LogisticRegression(class_weight='balanced', max_iter=1000).fit(X_train_imb, y_train_imb)
pred_balanced = logreg_balanced.predict(X_test_imb)

print("=== Model Standar (Mengabaikan Minoritas) ===")
print(classification_report(y_test_imb, pred_standard, target_names=["Mayoritas (0)", "Minoritas (1)"]))

print("\n=== Model Balanced (Sensitif terhadap Minoritas) ===")
print(classification_report(y_test_imb, pred_balanced, target_names=["Mayoritas (0)", "Minoritas (1)"]))

=== Model Standar (Mengabaikan Minoritas) ===
               precision    recall  f1-score   support

Mayoritas (0)       0.96      0.98      0.97       237
Minoritas (1)       0.50      0.31      0.38        13

     accuracy                           0.95       250
    macro avg       0.73      0.65      0.68       250
 weighted avg       0.94      0.95      0.94       250


=== Model Balanced (Sensitif terhadap Minoritas) ===
               precision    recall  f1-score   support

Mayoritas (0)       0.99      0.83      0.90       237
Minoritas (1)       0.23      0.92      0.36        13

     accuracy                           0.83       250
    macro avg       0.61      0.88      0.63       250
 weighted avg       0.95      0.83      0.88       250



## 3. Optimasi *Solver* dan Regularisasi (*L1, L2, Elastic-Net*)

Untuk menghitung bobot koefisien yang paling optimal, Regresi Logistik mengandalkan algoritma optimasi numerik yang disebut *solver*. Pemilihan *solver* sangat bergantung pada dimensi data dan jenis regularisasi yang diinginkan:

* **`lbfgs` (Bawaan):** Sangat efisien, namun hanya mendukung Regularisasi L2 (Penalti koefisien standar).
* **`liblinear`:** Sangat baik untuk dataset berskala kecil dan mendukung Regularisasi L1 (kemampuan seleksi fitur/menekan koefisien menjadi 0).
* **`saga`:** Diperuntukkan bagi dataset raksasa secara komputasi dan merupakan satu-satunya *solver* yang mendukung **Elastic-Net** (kombinasi hybrid antara penalti L1 dan L2).

In [27]:
# 1. Regresi Logistik dengan Regularisasi L1 (Seleksi Fitur via liblinear)
logreg_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.5, random_state=42)
logreg_l1.fit(X_train_m, y_train_m)

# 2. Regresi Logistik dengan Elastic-Net (Kombinasi L1 & L2 via saga)
# l1_ratio=0.5 berarti komposisi penalti adalah 50% L1 dan 50% L2
logreg_elastic = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5, max_iter=5000, random_state=42)
logreg_elastic.fit(X_train_m, y_train_m)

print("=== Komparasi Solvers & Penalties ===")
print(f"Skor Uji L1 (liblinear)      : {logreg_l1.score(X_test_m, y_test_m):.4f}")
print(f"Skor Uji Elastic-Net (saga)  : {logreg_elastic.score(X_test_m, y_test_m):.4f}")

# Menampilkan sifat selektif dari L1 (Mengecek berapa koefisien yang menjadi tepat nol)
print(f"\nSparsitas Koefisien L1 (Fitur yang dieliminasi menjadi 0): {np.sum(logreg_l1.coef_ == 0)}")

=== Komparasi Solvers & Penalties ===
Skor Uji L1 (liblinear)      : 0.9680
Skor Uji Elastic-Net (saga)  : 0.9600

Sparsitas Koefisien L1 (Fitur yang dieliminasi menjadi 0): 0


## 4. Batas Keputusan Non-Linier (*Non-linear Decision Boundaries*)

Secara fundamental, Regresi Logistik hanya mampu membentuk garis lurus (atau *hyperplane* datar) untuk memisahkan data. Jika himpunan data memiliki struktur sirkuler atau melengkung yang kompleks, model ini akan gagal beradaptasi secara absolut.

Namun, kemampuan model ini dapat diperluas untuk menangani batas non-linier dengan mengintegrasikannya dengan `PolynomialFeatures` di dalam sebuah `Pipeline`. Matriks fitur akan diekspansi secara matematis (misal: menambahkan $x^2$ atau persilangan fitur $x_1 x_2$) sehingga model linier mampu membentuk kurva hiperbolik atau parabola.

In [28]:
# Membuat dataset berbentuk lingkaran konsentris (Non-linier absolut)
X_circ, y_circ = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_circ, y_circ, random_state=42)

# 1. Regresi Logistik Standar (Pasti Gagal pada data sirkuler)
pipe_linear = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression())
])
pipe_linear.fit(X_train_c, y_train_c)

# 2. Regresi Logistik Ekstensi Polinomial Derajat 2 (Akan Membentuk Batas Melengkung)
pipe_poly = Pipeline([
    ("poly", PolynomialFeatures(degree=2)),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression())
])
pipe_poly.fit(X_train_c, y_train_c)

print("=== Ekstensi Non-Linier (Dataset Sirkuler) ===")
print(f"Akurasi Linier Standar (Hanya Garis Lurus) : {pipe_linear.score(X_test_c, y_test_c):.4f}")
print(f"Akurasi Ekstensi Polinomial (Kurva/Elips)  : {pipe_poly.score(X_test_c, y_test_c):.4f}")

=== Ekstensi Non-Linier (Dataset Sirkuler) ===
Akurasi Linier Standar (Hanya Garis Lurus) : 0.3333
Akurasi Ekstensi Polinomial (Kurva/Elips)  : 1.0000


## Kesimpulan

Regresi Logistik melampaui paradigma algoritma statistik sederhana dan dapat dikonfigurasi secara ekstensif:
1.  **Untuk Kasus Multikelas:** Pendekatan `multinomial` direkomendasikan karena memberikan kalibrasi probabilitas antar-kelas yang jauh lebih presisi dibandingkan `ovr`.
2.  **Untuk Deteksi Anomali/Minoritas:** Konfigurasi `class_weight='balanced'` membalikkan bias algoritma agar fokus menemukan subjek kelas krusial yang terabaikan.
3.  **Untuk Optimalisasi Dimensi Tinggi:** Integrasi parameter Regularisasi *L1* atau *Elastic-Net* via solver `saga` dan `liblinear` sangat menghemat memori.
4.  **Untuk Fleksibilitas Spasial:** Pembungkusan (*wrapping*) algoritma linier dengan fitur *Polinomial* dalam *Pipeline* menyuntikkan kapabilitas penyelesaian non-linier yang setara dengan model komputasi berat.